In [0]:
# CONSTANTES

CATALOG = 'big-data-spark-sql'
SCHEMA = 'enem'
VOLUME = f'/Volumes/{CATALOG}/{SCHEMA}/raw'

# PATHS
enem_2024_path = f"{VOLUME}/resultado_2024_sample.parquet"

In [0]:
enem_2024_df = (
    spark.read.parquet(enem_2024_path)
)
enem_2024_df.count()

In [0]:
enem_2024_df.display()

In [0]:
type(enem_2024_df)

In [0]:
enem_2024_df.printSchema()

In [0]:
enem_2024_df.show(truncate=False)

In [0]:
enem_2024_df.limit(5).show()

In [0]:
enem_2024_df.explain(mode="formatted")

In [0]:
# Qtde. de alunos que fizeram ENEM por estados
enem_2024_df.groupBy("SG_UF_PROVA").count().orderBy("count", ascending=False).show()

In [0]:
# Qtde de alunos que realizaram a prova sem problemas por estado.
# filter ou where
import pyspark.sql.functions as f


(enem_2024_df
 .filter(f.col("TP_STATUS_REDACAO") == 1)
 .groupBy("SG_UF_PROVA")
 .count()
 .orderBy("count", ascending=False)
 .limit(5)
 .show()
 )

In [0]:
(enem_2024_df
 .groupBy("SG_UF_PROVA")
 .agg(
     f.avg("NU_NOTA_LC").alias("MEDIA_NOTA_LC"),
     f.avg("NU_NOTA_MT").alias("MEDIA_NOTA_MT"),
     f.avg("NU_NOTA_REDACAO").alias("MEDIA_NOTA_REDACAO"),
 )
 .orderBy("MEDIA_NOTA_REDACAO", ascending=False)
 .limit(5)
 .show()
)

In [0]:
(enem_2024_df
 .filter(
     f.col("NU_NOTA_REDACAO") > 0
 )
 .groupBy("SG_UF_PROVA")
 .agg(
     f.avg("NU_NOTA_REDACAO").alias("MEDIA_NOTA_REDACAO"),
     f.std("NU_NOTA_REDACAO").alias("DESVIO_PADRAO_NOTA_REDACAO"),
     f.max("NU_NOTA_REDACAO").alias("MAIOR_NOTA_REDACAO"),
     f.min("NU_NOTA_REDACAO").alias("MENOR_NOTA_REDACAO"),
 )
 .orderBy("MEDIA_NOTA_REDACAO", ascending=False)
 .limit(5)
 .show()
)

In [0]:
# Qtde. de notas maximas na redação
(enem_2024_df
 .filter(
     f.col("NU_NOTA_REDACAO") >= 980
 )
 .groupBy("SG_UF_PROVA")
 .agg(
     f.count("NU_NOTA_REDACAO").alias("QTDE_NOTA_MAX_REDACAO")
 )
 .orderBy("QTDE_NOTA_MAX_REDACAO", ascending=False)
 .limit(5)
 .show()
)


In [0]:
# MEDIA EM MATEMATICA E LINGUAGENS DOS ALUNOS DO SUDESTE QUE ZERARAM A REDACAO, por estado
(enem_2024_df
 .filter(
    (f.col("NU_NOTA_REDACAO") == 0) & (f.col("SG_UF_PROVA").isin(["SP", "RJ", "ES", "MG"]))
 )
 .groupBy("SG_UF_PROVA")
 .agg(
    f.avg("NU_NOTA_LC").alias("MEDIA_NOTA_LC"),
    f.avg("NU_NOTA_MT").alias("MEDIA_NOTA_MT"),
 )
 .orderBy("MEDIA_NOTA_MT", ascending=False)
 .limit(5)
 .show()
)


In [0]:
# Usando sql

enem_2024_df.createOrReplaceTempView("vw_enem_2024")

query = """
SELECT
    SG_UF_PROVA,
    avg(NU_NOTA_LC) as MEDIA_NOTA_LC,
    avg(NU_NOTA_MT) as MEDIA_NOTA_MT
FROM vw_enem_2024
WHERE
    SG_UF_PROVA in ("SP", "RJ", "ES", "MG") AND
    NU_NOTA_REDACAO = 0
GROUP BY
    SG_UF_PROVA
ORDER BY
    MEDIA_NOTA_MT DESC
LIMIT 5
"""

spark.sql(query).show()
